# Agentic JEPA — Research Prototype

Neural architecture performing reasoning in a continuous latent space (S^(d-1)) with JEPA-style self-supervised prediction, tool-use, and latent-space planning/backtracking.

In [ ]:
# Cell 1: Setup & Imports
import sys
import os
import logging
import json
import random
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import defaultdict

# Add project root to path
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from config import AgenticJEPAConfig
from data.loader import (
    parse_codeact_trajectory, load_trajectories_from_jsonl,
    AgenticJEPADataset, create_dataloaders, collate_fn
)
from data.action_classifier import classify_action
from models.encoders import TextEncoder, create_ema_encoder
from models.afterstate_predictor import AfterstatePredictor
from models.slerp_fusion import GatedSLERPFusion
from models.value_head import LatentValueHead
from models.talker import Talker, validate_syntax
from training.losses import jepa_loss, value_loss, compute_total_loss
from training.curriculum import CurriculumController
from training.trainer import AgenticJEPATrainer
from inference.mcts import MCTSPlanner
from inference.backtracking import BacktrackingController
from utils.math_utils import l2_normalize, cosine_distance, safe_slerp
from utils.ema import update_ema

# Logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s'
)
logger = logging.getLogger('notebook')

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f'Using device: {device}')

# Config
config = AgenticJEPAConfig()
print(f'Config: d_model={config.d_model}, encoder={config.encoder_name}')
print(f'Device: {device}')

In [ ]:
# Cell 2: Data Loading
# ============================================================
# Option A: Load from a real CodeActInstruct JSONL file
# Uncomment and set the path to your dataset:
#
# DATASET_PATH = "/path/to/codeact_instruct.jsonl"
# trajectories = load_trajectories_from_jsonl(DATASET_PATH, max_count=config.num_trajectories)
#
# Option B: Generate synthetic trajectories for smoke-testing
# ============================================================

def make_synthetic_trajectories(n: int = 50):
    """Generate minimal synthetic trajectories for testing without real data."""
    import random
    actions = [
        "x = [i**2 for i in range(10)]\nprint(x)",
        "import math\nresult = math.sqrt(144)\nprint(result)",
        "data = {'a': 1, 'b': 2}\nprint(sum(data.values()))",
        "def fib(n):\n    return n if n<=1 else fib(n-1)+fib(n-2)\nprint(fib(10))",
        "import requests\nresponse = requests.get('http://example.com')\nprint(response.status_code)",
        "import subprocess\nresult = subprocess.run(['ls', '-la'], capture_output=True)\nprint(result.stdout)",
        "nums = [3, 1, 4, 1, 5, 9]\nprint(sorted(nums))",
        "text = 'hello world'\nprint(text.upper().split())",
    ]
    observations = [
        "[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]",
        "12.0",
        "3",
        "55",
        "200",
        "total 48\n-rw-r--r-- 1 user group 1234 Jan 1 config.py",
        "[1, 1, 3, 4, 5, 9]",
        "['HELLO', 'WORLD']",
    ]
    contexts = [
        "Write a Python script to compute squares of numbers 0-9.",
        "Compute the square root of 144 using the math module.",
        "Sum the values in a dictionary.",
        "Compute the 10th Fibonacci number recursively.",
        "Make an HTTP GET request to example.com.",
        "List all files in the current directory.",
        "Sort a list of integers.",
        "Convert 'hello world' to uppercase and split into words.",
    ]

    raw_trajs = []
    for i in range(n):
        idx = i % len(actions)
        n_steps = random.randint(1, 3)
        convos = [{"role": "user", "content": contexts[idx]}]
        for s in range(n_steps):
            act_idx = (idx + s) % len(actions)
            convos.append({"role": "assistant", "content": actions[act_idx]})
            convos.append({"role": "environment", "content": observations[act_idx]})
        raw_trajs.append({
            "conversations": convos,
            "reward": random.choice([0, 1])
        })
    return raw_trajs


raw_trajs = make_synthetic_trajectories(n=config.num_trajectories)
trajectories = [t for raw in raw_trajs if (t := parse_codeact_trajectory(raw)) is not None]

print(f'Loaded {len(trajectories)} trajectories')
print(f'Total steps: {sum(len(t.steps) for t in trajectories)}')
print(f'Success rate: {sum(t.reward for t in trajectories)/len(trajectories):.1%}')

# Build action vocabulary from all trajectories (for MCTS branch proposals)
action_vocabulary = list({step.action_text for t in trajectories for step in t.steps})
print(f'Action vocabulary size: {len(action_vocabulary)}')

# Create tokenizer (reuse from encoder — loaded in Cell 3)
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(config.encoder_name)
print(f'Tokenizer vocab size: {tokenizer.vocab_size}')

# Create dataloaders
train_loader, val_loader = create_dataloaders(
    trajectories,
    tokenizer=tokenizer,
    val_split=config.val_split,
    batch_size=config.batch_size,
    max_len=config.max_seq_len,
)
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

In [ ]:
# Cell 3: Model Initialization
trainer = AgenticJEPATrainer(config=config, device=device)

param_counts = trainer.count_parameters()
print('Parameter counts:')
for k, v in param_counts.items():
    print(f'  {k}: {v:,}')

# Quick sanity check: forward pass
test_texts = ["def hello(): print('world')", "x = 1 + 2"]
with torch.no_grad():
    h_test = trainer.encoder(test_texts)
    action_test = trainer.encoder(test_texts)
    as_test, n_steps_test, _ = trainer.predictor(h_test, action_test)
    obs_test = trainer.encoder(["output: 3", "output: hello"])
    h_fused_test, alpha_test = trainer.slerp_fusion(as_test, obs_test)
    v_test = trainer.value_head(h_fused_test)

print(f'\nSanity check:')
print(f'  Encoder output shape: {h_test.shape}')
print(f'  Afterstate shape: {as_test.shape}')
print(f'  ACT ponder steps: {n_steps_test.tolist()}')
print(f'  SLERP alpha: {alpha_test.squeeze().tolist()}')
print(f'  Value estimates: {v_test.tolist()}')
print(f'  h_t norm (should be ~1.0): {h_test.norm(dim=-1).tolist()}')
print(f'  as_t norm (should be ~1.0): {as_test.norm(dim=-1).tolist()}')
print(f'  h_fused norm (should be ~1.0): {h_fused_test.norm(dim=-1).tolist()}')

In [ ]:
# Cell 4: Stage 0 Training — Pure JEPA
# Only JEPA loss, internal actions only, no observations
print('=== STAGE 0: Pure JEPA Training ===')
print('Loss weights: λ_JEPA=1.0, λ_V=0.0, λ_ponder=0.0')
print('Data filter: internal actions only')

stage0_metrics = []
stage0_val_metrics = []
stage_transitions = []

for epoch in range(config.max_epochs_per_stage):
    if trainer.curriculum.state.current_stage != 0:
        stage_transitions.append(('0→1', trainer.global_step))
        print(f'\n>>> Stage transition 0→1 at step {trainer.global_step} <<<')
        break

    for batch in train_loader:
        metrics = trainer.train_step(batch)
        if metrics is not None:
            stage0_metrics.append(metrics)

    if (epoch + 1) % max(1, config.eval_every_n_steps // len(train_loader)) == 0:
        val_m = trainer.evaluate(val_loader)
        stage0_val_metrics.append({'epoch': epoch, **val_m})
        advanced = trainer.curriculum.check_transition(val_m)
        print(
            f'Epoch {epoch:3d} | Step {trainer.global_step:5d} | '
            f'JEPA_val={val_m["jepa_loss"]:.4f} | '
            f'Stage={trainer.curriculum.state.current_stage}'
        )
        if advanced:
            stage_transitions.append(('0→1', trainer.global_step))
            print(f'>>> Stage transition 0→1 at step {trainer.global_step} <<<')
            break

# Plot Stage 0 JEPA loss curve
if stage0_metrics:
    fig, ax = plt.subplots(figsize=(10, 4))
    steps = [m['step'] for m in stage0_metrics]
    jepa_losses = [m['loss_jepa'] for m in stage0_metrics]
    ax.plot(steps, jepa_losses, alpha=0.5, label='JEPA loss (train)', linewidth=0.8)

    # Smooth version
    if len(jepa_losses) > 20:
        window = max(1, len(jepa_losses) // 20)
        smoothed = np.convolve(jepa_losses, np.ones(window)/window, mode='valid')
        ax.plot(steps[:len(smoothed)], smoothed, label='JEPA loss (smoothed)', linewidth=2)

    if stage0_val_metrics:
        val_epochs = [m['epoch'] for m in stage0_val_metrics]
        val_losses = [m['jepa_loss'] for m in stage0_val_metrics]
        ax.plot(
            [steps[min(e * len(train_loader), len(steps)-1)] for e in val_epochs],
            val_losses, 'ro-', label='JEPA loss (val)', markersize=4
        )

    for label, step in stage_transitions:
        ax.axvline(step, color='green', linestyle='--', label=f'Transition {label}')

    ax.set_xlabel('Training step')
    ax.set_ylabel('JEPA loss (cosine distance)')
    ax.set_title('Stage 0: JEPA Loss Curve')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print(f'Stage 0 complete. Final JEPA val loss: {stage0_val_metrics[-1]["jepa_loss"]:.4f}')

In [ ]:
# Cell 5: Stage 1 Training — Mild Observations
print('=== STAGE 1: Mild Observations ===')
print('Loss weights: λ_JEPA=1.0, λ_V=0.01, λ_ponder=0.0')
print('Data filter: successful observations')

# Ensure we're in Stage 1 (or advance manually for testing)
if trainer.curriculum.state.current_stage == 0:
    print('Manually advancing to Stage 1 for demonstration...')
    trainer.curriculum.state.current_stage = 1
    trainer.curriculum.state.stage0_plateau_value = \
        stage0_val_metrics[-1]['jepa_loss'] if stage0_val_metrics else 0.5

stage1_metrics = []
stage1_val_metrics = []

for epoch in range(config.max_epochs_per_stage):
    if trainer.curriculum.state.current_stage != 1:
        print(f'\n>>> Stage transition 1→2 at step {trainer.global_step} <<<')
        stage_transitions.append(('1→2', trainer.global_step))
        break

    for batch in train_loader:
        metrics = trainer.train_step(batch)
        if metrics is not None:
            stage1_metrics.append(metrics)

    if (epoch + 1) % max(1, config.eval_every_n_steps // max(len(train_loader), 1)) == 0:
        val_m = trainer.evaluate(val_loader)
        stage1_val_metrics.append({'epoch': epoch, **val_m})
        advanced = trainer.curriculum.check_transition(val_m)
        print(
            f'Epoch {epoch:3d} | Step {trainer.global_step:5d} | '
            f'JEPA_val={val_m["jepa_loss"]:.4f} | '
            f'Alpha={val_m["mean_alpha"]:.3f} | '
            f'Stage={trainer.curriculum.state.current_stage}'
        )
        if advanced:
            stage_transitions.append(('1→2', trainer.global_step))
            print(f'>>> Stage transition 1→2 at step {trainer.global_step} <<<')
            break

# Plot Stage 1 metrics
if stage1_metrics:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    steps = [m['step'] for m in stage1_metrics]

    # JEPA loss
    axes[0].plot(steps, [m['loss_jepa'] for m in stage1_metrics], alpha=0.6, linewidth=0.8)
    axes[0].set_title('JEPA Loss')
    axes[0].set_xlabel('Step')
    axes[0].grid(True, alpha=0.3)

    # Value loss
    axes[1].plot(steps, [m['loss_value'] for m in stage1_metrics], alpha=0.6, linewidth=0.8, color='orange')
    axes[1].set_title('Value Loss')
    axes[1].set_xlabel('Step')
    axes[1].grid(True, alpha=0.3)

    # Mean alpha
    axes[2].plot(steps, [m['mean_alpha'] for m in stage1_metrics], alpha=0.6, linewidth=0.8, color='green')
    axes[2].axhline(config.stage1_alpha_threshold, color='red', linestyle='--', label=f'threshold={config.stage1_alpha_threshold}')
    axes[2].set_title('Mean SLERP α')
    axes[2].set_xlabel('Step')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    plt.suptitle('Stage 1: Mild Observations', fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 6: Stage 2 Training — Full Stochasticity
print('=== STAGE 2: Full Stochasticity ===')
print('Loss weights: λ_JEPA=1.0, λ_V=0.1, λ_ponder=0.01')
print('Data filter: all')

if trainer.curriculum.state.current_stage < 2:
    print('Manually advancing to Stage 2 for demonstration...')
    trainer.curriculum.state.current_stage = 2

stage2_metrics = []

for epoch in range(config.max_epochs_per_stage):
    for batch in train_loader:
        metrics = trainer.train_step(batch)
        if metrics is not None:
            stage2_metrics.append(metrics)

    if (epoch + 1) % max(1, config.eval_every_n_steps // max(len(train_loader), 1)) == 0:
        val_m = trainer.evaluate(val_loader)
        print(
            f'Epoch {epoch:3d} | Step {trainer.global_step:5d} | '
            f'JEPA={val_m["jepa_loss"]:.4f} | '
            f'Alpha={val_m["mean_alpha"]:.3f} | '
            f'Ponder={val_m["mean_ponder"]:.2f}'
        )

# Plot Stage 2 metrics
if stage2_metrics:
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))

    steps = [m['step'] for m in stage2_metrics]

    axes[0, 0].plot(steps, [m['loss_jepa'] for m in stage2_metrics], alpha=0.6, linewidth=0.8)
    axes[0, 0].set_title('JEPA Loss')
    axes[0, 0].set_xlabel('Step')
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].plot(steps, [m['loss_value'] for m in stage2_metrics], alpha=0.6, linewidth=0.8, color='orange')
    axes[0, 1].set_title('Value Loss')
    axes[0, 1].set_xlabel('Step')
    axes[0, 1].grid(True, alpha=0.3)

    axes[1, 0].plot(steps, [m['ponder_steps'] for m in stage2_metrics], alpha=0.6, linewidth=0.8, color='purple')
    axes[1, 0].axhline(config.act_max_steps, color='red', linestyle='--', label=f'K_max={config.act_max_steps}')
    axes[1, 0].set_title('ACT Ponder Steps')
    axes[1, 0].set_xlabel('Step')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].plot(steps, [m['mean_alpha'] for m in stage2_metrics], alpha=0.6, linewidth=0.8, color='green')
    axes[1, 1].set_title('Mean SLERP α')
    axes[1, 1].set_xlabel('Step')
    axes[1, 1].grid(True, alpha=0.3)

    plt.suptitle('Stage 2: Full Stochasticity', fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 7: Talker Training
# Freeze predictor and all other modules; train only the Talker
print('=== TALKER TRAINING ===')
print('Predictor frozen. Training Talker decoder with cross-entropy loss.')

talker = Talker(
    d_model=config.d_model,
    vocab_size=tokenizer.vocab_size,
    n_layers=config.talker_layers,
    n_heads=config.talker_heads,
    max_tokens=config.talker_max_tokens,
).to(device)

talker_losses = trainer.train_talker(
    talker=talker,
    train_loader=train_loader,
    tokenizer=tokenizer,
    max_epochs=10,
)

# Plot Talker cross-entropy loss
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(talker_losses) + 1), talker_losses, 'bo-', markersize=5)
ax.set_xlabel('Epoch')
ax.set_ylabel('Cross-entropy loss')
ax.set_title('Talker Training Loss')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Talker training complete. Final CE loss: {talker_losses[-1]:.4f}')

# Quick generation test
talker.eval()
with torch.no_grad():
    test_enc = trainer.encoder(["compute the mean of [1,2,3,4,5]"])
    test_as, _, _ = trainer.predictor(test_enc, test_enc)
    generated = talker.generate(test_as[:1], tokenizer, temperature=0.8)
print(f'\nGeneration test:\n  Generated: {repr(generated[:200])}')

In [ ]:
# Cell 8: Calibration
# Compute Δ_t distribution on validation set to set the delta_fatal threshold
print('=== CALIBRATION ===')

trainer.encoder.eval()
trainer.predictor.eval()
trainer.slerp_fusion.eval()
trainer.value_head.eval()

delta_values = []
alpha_by_type = defaultdict(list)  # 'internal' vs 'external'
ponder_by_step = defaultdict(list)

with torch.no_grad():
    for batch in val_loader:
        tau = batch['tau'].to(device)

        h_t = trainer.encoder(batch['pre_action_text'])
        action_embed = trainer.encoder(batch['action_text'])
        as_t, n_steps, _ = trainer.predictor(h_t, action_embed)

        obs_embed = trainer.encoder(batch['observation_text'])
        h_fused, alpha = trainer.slerp_fusion(as_t, obs_embed)

        v_as = trainer.value_head(as_t)
        v_fused = trainer.value_head(h_fused)

        deltas = (v_fused - v_as).cpu().numpy()
        delta_values.extend(deltas.tolist())

        # Track alpha by action type
        for i, t in enumerate(tau.cpu()):
            action_type = 'internal' if t.item() >= 0.9 else 'external'
            alpha_by_type[action_type].append(alpha[i].item())

        # Track ponder steps by step index
        for i, step_idx in enumerate(batch['step_idx']):
            ponder_by_step[step_idx].append(n_steps[i].item())

delta_array = np.array(delta_values)
print(f'Δ_t distribution over {len(delta_array)} steps:')
print(f'  Mean: {delta_array.mean():.4f}')
print(f'  Std:  {delta_array.std():.4f}')
print(f'  5th percentile: {np.percentile(delta_array, 5):.4f}')
print(f'  1st percentile: {np.percentile(delta_array, 1):.4f}')

# Recommend threshold at 5th percentile
recommended_threshold = np.percentile(delta_array, 5)
print(f'\nRecommended delta_fatal threshold (5th pct): {recommended_threshold:.4f}')
print(f'Current config.delta_fatal: {config.delta_fatal}')

In [ ]:
# Cell 9: Inference Demo
# Run MCTS + backtracking on a held-out example
print('=== INFERENCE DEMO ===')

trainer.encoder.eval()
trainer.predictor.eval()
trainer.slerp_fusion.eval()
trainer.value_head.eval()
talker.eval()

# Use a held-out trajectory
held_out_traj = trajectories[-1]  # Last trajectory (was in val set)
gt_actions = [step.action_text for step in held_out_traj.steps]
gt_observations = [step.observation_text for step in held_out_traj.steps]

# Simulated environment: return precomputed observations
obs_iter = iter(gt_observations)
def mock_environment(action_text: str) -> str:
    try:
        return next(obs_iter)
    except StopIteration:
        return "# end of trajectory"

# Create MCTS planner
planner = MCTSPlanner(
    encoder=trainer.encoder,
    predictor=trainer.predictor,
    slerp_fusion=trainer.slerp_fusion,
    value_head=trainer.value_head,
    talker=talker,
    tokenizer=tokenizer,
    action_vocabulary=action_vocabulary,
    device=device,
    n_branches=config.mcts_branches,
    delta_fatal=recommended_threshold,
    talker_retries=config.talker_retries,
)

print(f'Running inference on: "{held_out_traj.context[:80]}..."')
print(f'Ground-truth reward: {held_out_traj.reward}')
print()

traces = planner.run_trajectory(
    initial_context=held_out_traj.context,
    observation_fn=mock_environment,
    gt_actions=gt_actions,
    max_steps=min(len(held_out_traj.steps), 5),
)

# Print trace
print('\n--- LATENT REASONING TRACE ---')
for trace in traces:
    step = trace.get('step', '?')
    action_preview = (trace['selected_action'] or '')[:60]
    alpha = f"{trace['alpha']:.3f}" if trace['alpha'] is not None else 'N/A'
    delta = f"{trace['delta']:.4f}" if trace['delta'] is not None else 'N/A'
    syntax_ok = trace.get('syntax_valid', False)
    backtracked = trace.get('backtracked', False)

    print(f'Step {step}:')
    print(f'  Selected action: {action_preview!r}...')
    print(f'  SLERP α:        {alpha}')
    print(f'  Surprise Δ_t:   {delta}')
    print(f'  Syntax valid:   {syntax_ok}')
    print(f'  Backtracked:    {backtracked}')
    print()

In [ ]:
# Cell 10: Visualization
# Paper figures: α_t by observation type, Δ_t distribution, ACT ponder steps

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Plot 1: α_t distribution by observation type ---
ax = axes[0]
for obs_type, alphas in alpha_by_type.items():
    if alphas:
        ax.hist(alphas, bins=30, alpha=0.6, label=obs_type, density=True)

ax.set_xlabel('SLERP α (fusion weight)')
ax.set_ylabel('Density')
ax.set_title('α_t Distribution by Observation Type')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1])

# --- Plot 2: Δ_t (surprise) distribution ---
ax = axes[1]
ax.hist(delta_array, bins=50, alpha=0.7, color='steelblue', density=True)
ax.axvline(
    recommended_threshold, color='red', linestyle='--',
    label=f'delta_fatal={recommended_threshold:.3f}'
)
ax.axvline(0, color='black', linestyle='-', alpha=0.5, label='Δ=0')
ax.set_xlabel('Δ_t = V(h_{t+1}) - V(as_t)')
ax.set_ylabel('Density')
ax.set_title('Surprise Metric Δ_t Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

# --- Plot 3: ACT ponder steps vs step index ---
ax = axes[2]
step_indices = sorted(ponder_by_step.keys())
if step_indices:
    mean_ponders = [np.mean(ponder_by_step[s]) for s in step_indices]
    std_ponders = [np.std(ponder_by_step[s]) for s in step_indices]
    ax.plot(step_indices, mean_ponders, 'b-o', markersize=4, label='mean ponder steps')
    ax.fill_between(
        step_indices,
        [m - s for m, s in zip(mean_ponders, std_ponders)],
        [m + s for m, s in zip(mean_ponders, std_ponders)],
        alpha=0.2
    )
    ax.axhline(config.act_max_steps, color='red', linestyle='--', label=f'K_max={config.act_max_steps}')

ax.set_xlabel('Step index within trajectory')
ax.set_ylabel('ACT ponder steps')
ax.set_title('Ponder Steps vs Trajectory Position')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('Agentic JEPA — Evaluation Figures', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# --- Summary statistics ---
print('\n=== FINAL SUMMARY ===')
all_metrics = stage0_metrics + stage1_metrics + stage2_metrics
if all_metrics:
    final_jepa = [m['loss_jepa'] for m in all_metrics[-20:]]
    final_alpha = [m['mean_alpha'] for m in all_metrics[-20:]]
    final_ponder = [m['ponder_steps'] for m in all_metrics[-20:]]
    print(f'Total training steps: {trainer.global_step}')
    print(f'Final JEPA loss (last 20 steps mean): {np.mean(final_jepa):.4f}')
    print(f'Final mean α (last 20 steps mean):    {np.mean(final_alpha):.4f}')
    print(f'Final ponder steps (last 20 mean):     {np.mean(final_ponder):.2f}')

print(f'\nStage transitions: {stage_transitions}')
print(f'Recommended delta_fatal: {recommended_threshold:.4f}')